# OpenAI API Demo Notebook

This notebook demonstrates a practical Python workflow for the OpenAI API:

- setup and authentication
- listing available models
- text generation with different models
- structured JSON output
- embeddings
- moderation
- file upload + file input
- image generation
- optional audio transcription

It uses the official `openai` Python SDK and mainly relies on the **Responses API** for new projects.

## 1. Setup

Before running the notebook:

1. Create an API key in the OpenAI developer console.
2. Set it as an environment variable named `OPENAI_API_KEY`.
3. Install the latest SDK.

> Running these cells will make real API calls and may incur cost, except moderation which is free.

In [ ]:

# Uncomment if needed
# !pip install --upgrade openai pillow numpy matplotlib

In [ ]:

import os
import json
import base64
from pathlib import Path

import numpy as np
from openai import OpenAI

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise EnvironmentError(
        "OPENAI_API_KEY is not set. "
        "Set it in your environment before running this notebook."
    )

client = OpenAI(api_key=api_key)
print("Client initialized.")

## 2. Utility helpers

In [ ]:

def show_title(title: str):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

def preview(text: str, max_chars: int = 1200):
    text = str(text)
    print(text[:max_chars] + ("..." if len(text) > max_chars else ""))

def cosine_similarity(a, b):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom else 0.0

## 3. List a few available models

This is useful when you want to inspect what your project can access.

In [ ]:

show_title("Available models (first 25)")
models = client.models.list()

model_ids = sorted([m.id for m in models.data])
for mid in model_ids[:25]:
    print(mid)

print(f"\nTotal returned in this page: {len(models.data)}")

## 4. Simple text generation with different models

We compare a larger model and a smaller model on the same task.

In [ ]:

prompt = """
Write a concise explanation of what overfitting is in machine learning.
Give:
1. a one-sentence definition
2. one intuitive example
3. two practical ways to reduce it
"""

for model in ["gpt-5.4", "gpt-5.4-mini"]:
    show_title(f"Response from {model}")
    response = client.responses.create(
        model=model,
        input=prompt,
    )
    preview(response.output_text)

## 5. Reasoning-style task

Some models support reasoning controls. This example uses a reasoning model configuration for a small algorithmic task.

In [ ]:

reasoning_response = client.responses.create(
    model="gpt-5",
    reasoning={"effort": "low"},
    input=(
        "A sorted array has length 1,000,000. "
        "What is the time complexity of binary search, and about how many "
        "comparisons are needed in the worst case? Answer briefly."
    ),
)

show_title("Reasoning example")
preview(reasoning_response.output_text)

## 6. Structured JSON output

This pattern is useful for classification, extraction, routing, and API pipelines.

Here we ask the model to return a JSON object that matches a schema.

In [ ]:

schema = {
    "name": "ticket_router",
    "schema": {
        "type": "object",
        "properties": {
            "department": {
                "type": "string",
                "enum": ["billing", "technical_support", "sales", "other"]
            },
            "priority": {
                "type": "string",
                "enum": ["low", "medium", "high"]
            },
            "summary": {"type": "string"},
            "needs_human": {"type": "boolean"}
        },
        "required": ["department", "priority", "summary", "needs_human"],
        "additionalProperties": False
    }
}

ticket_text = (
    "Hi, I was charged twice for my subscription this month. "
    "Please fix it and refund the extra payment."
)

structured = client.responses.create(
    model="gpt-5.4-mini",
    input=f"Classify this support ticket: {ticket_text}",
    text={
        "format": {
            "type": "json_schema",
            "name": schema["name"],
            "schema": schema["schema"],
            "strict": True,
        }
    },
)

show_title("Structured JSON")
parsed = json.loads(structured.output_text)
print(json.dumps(parsed, indent=2))

## 7. Embeddings

Embeddings map text into vectors. A common use is semantic similarity.

In [ ]:

texts = [
    "cheap flights to paris",
    "inexpensive flights to paris",
    "how to cook pasta",
]

emb = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts,
)

vectors = [item.embedding for item in emb.data]

show_title("Embedding dimensions")
print(len(vectors[0]))

show_title("Pairwise cosine similarities")
for i in range(len(texts)):
    for j in range(i + 1, len(texts)):
        sim = cosine_similarity(vectors[i], vectors[j])
        print(f"{texts[i]!r}  vs  {texts[j]!r}  ->  {sim:.4f}")

## 8. Moderation

Moderation is useful before sending user content into downstream generation pipelines.

In [ ]:

moderation = client.moderations.create(
    model="omni-moderation-latest",
    input="I am angry and I want to hurt someone."
)

show_title("Moderation result")
print("Model:", moderation.model)
print("Flagged:", moderation.results[0].flagged)
print(json.dumps(moderation.results[0].categories.model_dump(), indent=2))

## 9. File upload and file input

This section creates a small local text file, uploads it, and then asks the model questions about the uploaded file.

For larger knowledge tasks, retrieval / file search is often a better pattern than passing very large files directly.

In [ ]:

sample_text = """
Course: AI for Engineers

Week 1 topics:
- linear regression
- train/validation/test split
- gradient descent

Week 2 topics:
- logistic regression
- classification metrics
- confusion matrix

Week 3 topics:
- embeddings
- transformers
- RAG
""".strip()

sample_path = Path("sample_course_outline.txt")
sample_path.write_text(sample_text, encoding="utf-8")
print(f"Wrote {sample_path.resolve()}")

In [ ]:

uploaded_file = client.files.create(
    file=open(sample_path, "rb"),
    purpose="user_data",
)

show_title("Uploaded file")
print("File id:", uploaded_file.id)
print("Filename:", uploaded_file.filename)
print("Purpose:", uploaded_file.purpose)

In [ ]:

file_response = client.responses.create(
    model="gpt-5.4-mini",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "Summarize the uploaded file and list the topics by week."
                },
                {
                    "type": "input_file",
                    "file_id": uploaded_file.id,
                },
            ],
        }
    ],
)

show_title("Model answer over uploaded file")
preview(file_response.output_text)

## 10. Image generation

This uses the Images API and saves the returned image locally.

In [ ]:

image_prompt = (
    "A clean educational illustration of a neural network with labeled input, "
    "hidden, and output layers, modern flat design, white background."
)

image_result = client.images.generate(
    model="gpt-image-2",
    prompt=image_prompt,
)

image_bytes = base64.b64decode(image_result.data[0].b64_json)
image_path = Path("openai_api_demo_image.png")
image_path.write_bytes(image_bytes)

show_title("Saved generated image")
print(image_path.resolve())

In [ ]:

from IPython.display import Image, display
display(Image(filename="openai_api_demo_image.png"))

## 11. Optional: audio transcription

Run this only if you have an audio file such as `speech.mp3`.

In [ ]:

audio_path = Path("speech.mp3")

if audio_path.exists():
    transcript = client.audio.transcriptions.create(
        model="gpt-4o-transcribe",
        file=open(audio_path, "rb"),
    )
    show_title("Transcription")
    preview(transcript.text if hasattr(transcript, "text") else transcript)
else:
    print("No speech.mp3 found in the working directory. Skipping.")

## 12. Notes

- For new builds, prefer the **Responses API**.
- Use **smaller models** for lower cost and latency.
- Use **embeddings** for semantic search and retrieval pipelines.
- Use **moderation** as a safety layer.
- Use **file uploads** when the model needs direct access to a document.
- Use the **Images API** when you want direct image generation from Python.